# Veggieezee — Next-Day Price Validation Report

**Project:** Vegetable price prediction (Kalimati Market, Nepal)  
**Model:** XGBoost regressor (`nepal_veg_price_xgboost.pkl`)  
**Purpose:** Show evidence that our deployed model follows the validation method requested in class:

> Use **today’s** market data to predict **tomorrow’s** price, then compare the forecast with **tomorrow’s actual** Kalimati price.

This notebook is written so a reviewer (friend / sir) can read the method first, run the cells, and capture **logs + charts** for the DEV report.

---

### What this notebook proves

| Question | Answer in this notebook |
|----------|-------------------------|
| Do we retrain every day? | **No** — we use the existing trained model. |
| What data do we use? | **30 days** of Kalimati prices; commodity names in **Nepali**. |
| How do we validate? | **15 next-day forecasts** (days 16–30): today → predict tomorrow → compare actual. |
| How are errors reported? | **MAE, RMSE, MAPE** + **two PNG charts** (forecast + error analysis). |

---

### Files required (in repo / `predict/ml/`)

| File | Role |
|------|------|
| `kalimati_vegetable_prices_last_30_days.csv` | 30-day Kalimati export (Nepali names) |
| `validate_next_day.py` | Validation + plotting script |
| `kalimati_nepali_to_english.csv` | Nepali ↔ English map (auto-built if missing) |
| `../../models/nepal_veg_price_xgboost.pkl` | Trained model |
| `../../models/nepal_veg_label_encoder.pkl` | Vegetable label encoder |

**Google Colab:** Run the **git clone** cell below (full repo from GitHub). No need to upload the notebook alone.

## 1. Validation methodology (sir’s flow)

We implement **walk-forward, one-step-ahead** forecasting:

1. **Day N (today):** Collect today’s Kalimati min / max / average price for each vegetable (and all prior days in the dataset for lag features).
2. **Predict Day N+1 (tomorrow):** Feed features into the **already trained** XGBoost model. The model outputs a value in **log space** (`log1p(price)`); we convert back to rupees with `expm1`.
3. **Day N+1 arrives:** Read the **actual** average price from the dataset.
4. **Compare:** Record `predicted_npr`, `actual_npr`, and error.
5. **Repeat** for each validation day and each vegetable that exists in our training taxonomy.

**Timeline on our 30-day file:**

- **Days 1–15:** History only (build lags; not used as forecast targets in the score).
- **Days 16–30:** For each day, predict the **next** calendar day and score against actuals → **15 validation days**.

**Important:** We do **not** retrain the model during this loop. That matches real use: train once, predict daily with new data.

## 2. Environment setup

**Colab order:** (1) pip install → (2) git clone → (3) imports / Django setup → run rest top to bottom.

In [ ]:
# Run once in Colab or local venv
%pip install -q pandas numpy matplotlib scikit-learn xgboost joblib cloudscraper django openpyxl

In [ ]:
# Option A (recommended): clone full repo from GitHub
import os
from pathlib import Path

REPO = Path('/content/ct654-veggieezee')
if not (REPO / 'veggieezee' / 'manage.py').is_file():
    if REPO.exists():
        !rm -rf {REPO}
    !git clone https://github.com/7n5aj/ct654-veggieezee.git {REPO}

PROJECT_ROOT = REPO / 'veggieezee'

# Update if repo was cloned earlier (gets latest validate_next_day.py)
if (PROJECT_ROOT / 'manage.py').is_file():
    !cd {REPO} && git pull origin main

print('manage.py exists:', (PROJECT_ROOT / 'manage.py').is_file())
print('validate_next_day.py:', (PROJECT_ROOT / 'predict' / 'ml' / 'validate_next_day.py').is_file())
print('model exists:', (PROJECT_ROOT / 'models' / 'nepal_veg_price_xgboost.pkl').is_file())
print('30-day CSV exists:', (PROJECT_ROOT / 'predict' / 'ml' / 'kalimati_vegetable_prices_last_30_days.csv').is_file())

In [ ]:
import os
import sys
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

COLAB_ROOT = Path('/content/ct654-veggieezee/veggieezee')

def _find_project_root() -> Path:
    if COLAB_ROOT.is_dir() and (COLAB_ROOT / 'manage.py').is_file():
        return COLAB_ROOT
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'manage.py').is_file():
            return candidate
    raise FileNotFoundError('Run the git clone cell above first.')

PROJECT_ROOT = _find_project_root()
ML_DIR = PROJECT_ROOT / 'predict' / 'ml'
SCRIPT = ML_DIR / 'validate_next_day.py'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'veggieezee.settings')

import django
django.setup()

# Load script by path (predict.ml is not a Python package on Colab)
spec = importlib.util.spec_from_file_location('validate_next_day', SCRIPT)
validate_next_day = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validate_next_day)

DATA_PATH = validate_next_day.DATA_PATH
RESULTS_CSV = validate_next_day.RESULTS_CSV
CHART_PATH = validate_next_day.CHART_PATH
ERRORS_CHART_PATH = validate_next_day.ERRORS_CHART_PATH
TRAIN_DAYS = validate_next_day.TRAIN_DAYS
_load_kalimati_30d = validate_next_day._load_kalimati_30d
_load_nepali_to_english = validate_next_day._load_nepali_to_english
run_validation = validate_next_day.run_validation
plot_validation_charts = validate_next_day.plot_validation_charts

print('Project root:', PROJECT_ROOT)
print('Script:      ', SCRIPT)
print('Script exists:', SCRIPT.is_file())

## 3. Dataset description (for report)

**Source:** Kalimati Fruit and Vegetable Market — daily price API / export  
**Target column:** `avg_price_npr` (average price in Nepalese Rupees)  
**Commodity labels:** **Nepali** (`commodity` column), as required for local reporting  
**Window:** 30 consecutive calendar days

In [ ]:
df = _load_kalimati_30d()

date_min, date_max = df['date'].min(), df['date'].max()
n_days = df['date'].nunique()
n_rows = len(df)
n_commodities = df['commodity'].nunique()

summary = pd.DataFrame({
    'Metric': [
        'Start date', 'End date', 'Calendar days', 'Total rows',
        'Unique commodities (Nepali)', 'Price column',
    ],
    'Value': [
        str(date_min.date()), str(date_max.date()), n_days, n_rows,
        n_commodities, 'avg_price_npr',
    ],
})
display(summary)

print('\nSample rows (Nepali commodity names):')
display(df.head(8))

## 4. Model note — log transform

During training (`nepal_xgboost_training.ipynb`), the target was:

$$y = \log(1 + \text{price})$$

So at prediction time we must **inverse-transform**:

$$\text{price} = e^y - 1 = \texttt{expm1}(y)$$

If this step is skipped, plots show values around **4–6** instead of **50–400** NPR and look completely wrong. This notebook applies `np.expm1` correctly.

In [ ]:
import joblib

model_path = PROJECT_ROOT / 'models' / 'nepal_veg_price_xgboost.pkl'
encoder_path = PROJECT_ROOT / 'models' / 'nepal_veg_label_encoder.pkl'

model = joblib.load(model_path)
encoder = joblib.load(encoder_path)

print(f'Model file:     {model_path.name}')
print(f'Trees:          {getattr(model, "n_estimators", "?")}')
print(f'Vegetable classes in encoder: {len(encoder.classes_)}')
print('Target at train time: log1p(average price)')
print('Inference: price_npr = expm1(model.predict(features))')

## 5. Nepali commodity names

The model was trained on English/normalized class names internally. For the report we keep **Nepali** in the results table. Mapping is built from Kalimati **EN** and **NP** API lists (same order, same prices).

In [ ]:
nepali_map = _load_nepali_to_english()
map_df = pd.DataFrame([{'nepali': k, 'english': v} for k, v in list(nepali_map.items())[:12]])
print(f'Total mapped pairs: {len(nepali_map)}')
display(map_df)

## 6. Run validation (screenshot this section for evidence)

Execute the cell below. **Copy the printed log** into your report or take a **screenshot** showing MAE / RMSE / MAPE.

In [ ]:
results = run_validation()

mae = results['abs_error_npr'].mean()
rmse = np.sqrt((results['error_npr'] ** 2).mean())
mape = results['pct_error'].mean()

metrics = pd.DataFrame({
    'Metric': ['MAE (NPR)', 'RMSE (NPR)', 'MAPE (%)', 'Forecast pairs', 'Validation days'],
    'Value': [
        round(mae, 2), round(rmse, 2), round(mape, 2),
        len(results), results['predict_date'].nunique(),
    ],
})
display(metrics)

## 7. Sample predictions (Nepali names)

A few rows showing **today → predict tomorrow → actual**.

In [ ]:
sample = results.sort_values('abs_error_npr').head(10)[
    ['nepali_name', 'today', 'predict_date', 'predicted_npr', 'actual_npr', 'abs_error_npr', 'pct_error']
]
display(sample)

print('\nLargest errors (for discussion / limitations):')
display(results.sort_values('abs_error_npr', ascending=False).head(5)[
    ['nepali_name', 'predict_date', 'predicted_npr', 'actual_npr', 'abs_error_npr']
])

## 8. Graphs (required figures)

Creates **two PNG files** in `predict/ml/`:

1. **`validation_predicted_vs_actual.png`** — forecast vs actual (Nepali names in legend)
2. **`validation_error_analysis.png`** — error histograms, daily mean error, scatter plot

Include both in the report under **Real-world validation**.

In [ ]:
%matplotlib inline

# Builds both PNGs (downloads Nepali font on first run if needed)
results = plot_validation_charts(results)

print('Saved charts:')
print(' 1)', CHART_PATH)
print(' 2)', ERRORS_CHART_PATH)

if CHART_PATH.is_file():
    display(Image(filename=str(CHART_PATH), width=900))
else:
    print('Missing:', CHART_PATH)

if ERRORS_CHART_PATH.is_file():
    display(Image(filename=str(ERRORS_CHART_PATH), width=900))
else:
    print('Missing:', ERRORS_CHART_PATH)

## 9. How to interpret results

**Chart 1 — Predicted vs actual**
- **Left:** Daily average predicted vs actual (all vegetables).
- **Right:** Three sample vegetables (Nepali names) — lines should move together.

**Chart 2 — Error analysis**
- **Top-left:** How often each absolute error (NPR) occurs.
- **Top-right:** Distribution of percentage error per forecast.
- **Bottom-left:** Mean absolute error per validation day.
- **Bottom-right:** Scatter — points near the diagonal line = good predictions.

**Typical scores:** MAE ~8 NPR, MAPE ~8% on this 30-day window (after `expm1` inverse transform).

---

## 10. Limitations (include in report)

1. **Short window:** Only 30 days; first 15 days are context, only 15 days are scored.
2. **Region:** Kalimati wholesale market only — not all of Nepal.
3. **No daily retrain:** Production could retrain weekly/monthly; we did not do that here.
4. **Market shocks:** Sudden supply shocks, festivals, or transport issues are only partly captured by seasonal features.
5. **Not all Nepali commodities** map to a trained class (~67/93 in a typical run).

---

## 11. Evidence checklist for submission

Give sir / marker:

- [ ] This notebook (`.ipynb`) or PDF export with **outputs visible**
- [ ] Screenshot of **Section 6** log (MAE, RMSE, MAPE)
- [ ] `validation_predicted_vs_actual.png`
- [ ] `validation_error_analysis.png`
- [ ] `validation_next_day_results.csv` (optional appendix)
- [ ] One paragraph citing **dataset source + size** (Section 3 table)
- [ ] Training notebook (`nepal_xgboost_training.ipynb`) **only** if asked how the model was built

---

**Outputs saved next to this notebook:**

- `validation_next_day_results.csv`
- `validation_predicted_vs_actual.png`
- `validation_error_analysis.png`